# Aurora inference & fine-tuning locally

This notebook explains the workflow and executes **Microsoft Aurora** inference and fine-tuning scripts via `runpy`.

## Files used in this workshop

We use the following resources:

1. [notebooks/0_aurora_workshop_local.ipynb](0_aurora_workshop_local.ipynb) *(this notebook)* – explains the workflow and runs scripts via `runpy`:
   - <mark>Run this notebook with a virtual environment containing the project dependencies</mark>
2. [setup/components/inference/](../setup/components/inference/) - contains Aurora inference logic:
   - [main.py](../setup/components/inference/main.py): a script with a CLI interface for running a simple inference loop
3. [setup/components/finetuning/](../setup/components/finetuning/) - contains Aurora fine-tuning logic:
   - [main.py](../setup/components/finetuning/main.py): a script with a CLI interface for running a simple fine-tuning loop
   - [utils.py](../setup/components/finetuning/utils.py): fine-tuning helper and utils logic
4. [setup/components/common/utils.py](../setup/components/common/utils.py) - contains Aurora helper logic, including:
   - Loading model checkpoints in train or eval mode
   - Loading data on disk into `aurora.Batch` objects for inference and fine-tuning
   - Converting `aurora.Batch` objects into `xarray.Dataset` objects for analysis and writing of data

#### Notes

Inference and fine-tuning scripts in [setup/components/](../setup/components/)*/main.py will work in local and remote environments provided the hardware and dependencies required to run Aurora are present in each. Resources in [setup/notebooks/](../setup/notebooks/) and [setup/common/](../setup/common), respectively, are largely for workshop setup including data and pre-trained model download and remote asset registration.

In [ ]:
import runpy
import sys
from pathlib import Path

import mlflow
import numpy as np
import xarray as xr
import yaml
from huggingface_hub import hf_hub_download

sys.path.insert(0, str(Path.cwd().parent.resolve()))
from setup.common.constants import (
    END_DATETIME,
    FINETUNE_CKPT_FILENAME,
    FINETUNE_CONFIG_PATH,
    FINETUNE_LOSS_FILENAME,
    FINETUNE_MODULE,
    FINETUNE_OUT_DIR,
    FINETUNE_PRED_FILENAME,
    HF_REPOSITORY,
    INFERENCE_CONFIG_PATH,
    INFERENCE_MODULE,
    INFERENCE_OUT_DIR,
    INFERENCE_PREDS_FILENAME,
    LOCAL_DATA_PATH,
    MODEL_FILENAME,
    OUTPUTS_DIR,
    START_DATETIME,
)
from setup.components.common.models import FinetuneConfig, InferenceConfig

Retrieve the model and input data paths and create output directories.

#### Notes

The input data and model checkpoints are expected to have been pre-loaded per parameters defined in [setup/common/constants.py](../setup/common/constants.py) with the [setup/notebooks/load_era5_local.ipynb](../setup/notebooks/load_era5_local.ipynb) and [setup/notebooks/load_model.ipynb](../setup/notebooks/load_model.ipynb) notebooks, respectively.

The following new directories and files will be created either now or by notebook end:

```md
aurora-introductory-workshop/
└── outputs/
    ├── inference/
    |       └── predictions.nc: forecasts generated in inference with the pre-trained model and ERA5 data
    └── finetuning/
            ├── loss.npy: loss history (loss values at each step) of fine-tuning
            ├── prediction.nc: last forecast generated in inference with the fine-tuned model and ERA5 data
            └── finetuned.ckpt: fine-tuned model checkpoint
```

In [ ]:
# downloads checkpoint if not already present in local cache
MODEL_PATH = hf_hub_download(repo_id=HF_REPOSITORY, filename=MODEL_FILENAME)
if not LOCAL_DATA_PATH.exists():
    print(
        f"Data not found at {LOCAL_DATA_PATH}, run "
        "setup/notebooks/load_era5_local.ipynb.",
    )

print(f"Using assets: model={MODEL_PATH}, data={LOCAL_DATA_PATH}")
COMMON_INPUTS = [
    "--model", MODEL_PATH,
    "--data", str(LOCAL_DATA_PATH),
    # initial state timestamp below and that -6 hours must exist in the data
    "--start_datetime", START_DATETIME,
]
INFERENCE_OUT_DIR.mkdir(exist_ok=True, parents=True)
FINETUNE_OUT_DIR.mkdir(exist_ok=True, parents=True)

## Inference jobs

Here, we'll run the inference and evaluation script using different data:

- Generated synthetic test data comprising a low resolution tensor of random float values
- Real, pre-loaded ERA5 data over the 2025-01-01T00 to 2025-01-31T18 period

After inference, the final prediction is compared to its corresponding ground truth. We calculate global difference and RMSE, logging the plot and value, respectively. Both are logged with [MLflow](https://mlflow.org/), an open-source machine learning lifecycle framework [integrated in AML](https://learn.microsoft.com/en-us/azure/machine-learning/concept-mlflow?view=azureml-api-2).

First, load the [inference configs defined in YAML](inference_configs.yaml) into a dictionary.

In [ ]:
with INFERENCE_CONFIG_PATH.open("r") as f:
    inference_configs = yaml.safe_load(f)

Then, specifying the name of a config block defined in the YAML, run the inference script.

For the script, see [setup/components/inference/main.py](../setup/components/inference/main.py).

In [ ]:
config_name = input(f"Enter a config name e.g. {next(iter(inference_configs))}").strip()
try:
    cfg = InferenceConfig.model_validate(inference_configs[config_name])
except KeyError as e:
    msg = f"Config not found: name={config_name}"
    raise ValueError(msg) from e

mlflow.set_tracking_uri((OUTPUTS_DIR / "mlruns").as_uri())
inference_ds_path = INFERENCE_OUT_DIR / INFERENCE_PREDS_FILENAME
sys.argv = [
    "main",
    *COMMON_INPUTS,
    "--config", cfg.model_dump_json(),
    "--predictions", str(inference_ds_path),
]
runpy.run_module(INFERENCE_MODULE, run_name="__main__", alter_sys=True)
mlflow.end_run()

## Fine-tuning jobs

Here, we'll run the fine-tuning script with different data:

- Generated synthetic test data comprising a low resolution tensor of random float values
- Real, pre-loaded ERA5 data over the 2025-01-01T00 to 2025-01-31T18 period

First, load the [fine-tuning configs defined in YAML](finetune_configs.yaml) into a dictionary.

In [ ]:
with FINETUNE_CONFIG_PATH.open("r") as f:
    finetune_configs = yaml.safe_load(f)

Then, specifying the name of a config block defined in the YAML, run the fine-tuning script.

For the script, see [setup/components/finetuning/main.py](../setup/components/finetuning/main.py).

In [ ]:
config_name = input(f"Enter a config name e.g. {next(iter(finetune_configs))}").strip()
try:
    cfg = FinetuneConfig.model_validate(finetune_configs[config_name])
except KeyError as e:
    msg = f"Config not found: name={config_name}"
    raise ValueError(msg) from e

finetune_ds_path = FINETUNE_OUT_DIR / FINETUNE_PRED_FILENAME
finetune_loss_arr_path = FINETUNE_OUT_DIR / FINETUNE_LOSS_FILENAME
sys.argv = [
    "main",
    *COMMON_INPUTS,
    # below timestamp only possibly used as a training target
    "--end_datetime", END_DATETIME,
    "--config", cfg.model_dump_json(),
    "--loss", str(finetune_loss_arr_path),
    "--prediction", str(finetune_ds_path),
    "--checkpoint", str(FINETUNE_OUT_DIR / FINETUNE_CKPT_FILENAME),
]
runpy.run_module(FINETUNE_MODULE, run_name="__main__", alter_sys=True)

## [Optional] Plotting and evaluating fine-tuning results

Here, we'll load and plot the products of inference and fine-tuning.

In [ ]:
inference_ds = xr.open_dataset(inference_ds_path)
finetune_ds = xr.open_dataset(finetune_ds_path)
loss_arr = np.load(finetune_loss_arr_path)

Try plotting the downloaded data.

In [ ]:
import matplotlib.pyplot as plt

## [Optional] Create an Aurora Batch

Using the produced forecast NetCDFs, try to create an `aurora.Batch` object from the data within.

In [ ]:
from aurora import Batch